In [1]:
import requests, urllib.parse

base = "https://mapy.geoportal.gov.pl/wss/service/rcn"

url = base + "?" + urllib.parse.urlencode({
    "SERVICE": "WFS", "VERSION": "2.0.0", "REQUEST": "GetFeature",
    "TYPENAMES": "budynki", "COUNT": 100, "STARTINDEX": 0,
})
r = requests.get(url)
print(len(r.content) / 100, "bytes/feature (avg)")
print(6_233_436 * len(r.content) / 100 / 1e9, "GB estimated total")

2539.64 bytes/feature (avg)
15.83068340304 GB estimated total


In [ ]:
import geopandas as gpd
import pandas as pd
import requests
from xml.etree import ElementTree as ET
from urllib.parse import urlencode

base = "https://mapy.geoportal.gov.pl/wss/service/rcn"

def count_features(typename):
    url = base + "?" + urlencode({
        "SERVICE": "WFS", "VERSION": "2.0.0", "REQUEST": "GetFeature",
        "TYPENAMES": typename, "RESULTTYPE": "hits"
    })
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    root = ET.fromstring(r.content)
    return int(root.attrib.get("numberMatched", -1))

def download_all(typename, page_size=1000):   # start conservative, raise once you confirm no server-side cap
    total = count_features(typename)
    print(f"{typename}: {total} features total")
    frames, start = [], 0
    while start < total:
        url = base + "?" + urlencode({
            "SERVICE": "WFS", "VERSION": "2.0.0", "REQUEST": "GetFeature",
            "TYPENAMES": typename, "COUNT": page_size, "STARTINDEX": start,
            "SORTBY": "gml_id",
        })
        gdf = gpd.read_file(url)
        if gdf.empty:
            break
        frames.append(gdf)
        start += page_size
        print(f"  {start}/{total}")
    return pd.concat(frames, ignore_index=True) if frames else gpd.GeoDataFrame()

def download_powiat(typename, teryt, page_size=5000):
    n = count_features_filtered(typename, teryt)   # hits request with the where/filter applied
    frames, start = [], 0
    while start < n:
        url = base + "?" + urlencode({
            "SERVICE": "WFS", "VERSION": "2.0.0", "REQUEST": "GetFeature",
            "TYPENAMES": typename, "COUNT": page_size, "STARTINDEX": start,
        })  # + your working filter mechanism, once confirmed
        gdf = gpd.read_file(url)
        if gdf.empty:
            break
        frames.append(gdf)
        start += page_size
    result = pd.concat(frames, ignore_index=True) if frames else gpd.GeoDataFrame()
    if len(result) != n:
        print(f"WARNING teryt={teryt}: expected {n}, got {len(result)}")
    return result

#budynki = download_all("budynki")
#budynki_0261 = budynki[budynki["teryt"] == "0261"]   # filter locally once you have everything

In [5]:
for _ in range(5):
    print(count_features("budynki"))
    import time; time.sleep(5)

6233436
6233436
6233436


KeyboardInterrupt: 